In [1]:
# Cell 1: config

MODEL_PATH = "/data/aneesh/pruning/oss_120b_mxfp4"  
TEXTS_JSON_PATH = "./texts_new.json"
OUTPUT_JSON_PATH = "./mappings/expert_mapping_120b_mxfp4.json"


In [2]:
import os 
os.environ["CUDA_VISIBLE_DEVICES"] = "3"


import json
import warnings
from pathlib import Path

import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

warnings.filterwarnings("ignore")


/data/aneesh/env_aime/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Cell 3: recorder (robust router-output parsing)

class ExpertPatternRecorder:
    def __init__(self, model, tokenizer, model_path, texts_json_path):
        self.model = model
        self.tokenizer = tokenizer
        self.model_path = model_path
        self.texts_json_path = texts_json_path
        self.hooks = []

        self.layers = self.model.model.layers
        self.num_layers = len(self.layers)
        self.num_experts = int(self.model.config.num_local_experts)

        self.reset()

    def reset(self):
        self.selection_counts = torch.zeros(self.num_layers, self.num_experts, dtype=torch.long)
        self.selection_prob_sums = torch.zeros(self.num_layers, self.num_experts, dtype=torch.float64)
        self.token_counts = torch.zeros(self.num_layers, dtype=torch.long)

        self.num_texts = 0
        self.text_lengths = []

    def _input_device(self):
        return next(self.model.parameters()).device

    def setup_hooks(self):
        self.remove_hooks()
        for layer_idx, layer in enumerate(self.layers):
            hook = layer.mlp.router.register_forward_hook(self._make_hook(layer_idx))
            self.hooks.append(hook)

    def remove_hooks(self):
        for h in self.hooks:
            h.remove()
        self.hooks = []

    def _extract_router_tensors(self, output):
        scores = None
        indices = None

        if isinstance(output, dict):
            for k in ["router_scores", "scores", "topk_scores", "routing_weights"]:
                if k in output and torch.is_tensor(output[k]):
                    scores = output[k]
                    break
            for k in ["router_indices", "indices", "topk_indices", "selected_experts"]:
                if k in output and torch.is_tensor(output[k]):
                    indices = output[k]
                    break

        elif isinstance(output, (tuple, list)):
            tensors = [x for x in output if torch.is_tensor(x)]

            for x in tensors:
                if x.dtype in (torch.int8, torch.int16, torch.int32, torch.int64, torch.long):
                    if indices is None:
                        indices = x
                elif x.dtype.is_floating_point:
                    if scores is None:
                        scores = x

            if (scores is None or indices is None) and len(tensors) >= 2:
                for i in range(len(tensors) - 1):
                    a, b = tensors[i], tensors[i + 1]
                    if a.dtype.is_floating_point and b.dtype in (
                        torch.int8, torch.int16, torch.int32, torch.int64, torch.long
                    ):
                        scores, indices = a, b
                        break

        else:
            for attr in ["router_scores", "scores", "topk_scores", "routing_weights"]:
                if hasattr(output, attr) and torch.is_tensor(getattr(output, attr)):
                    scores = getattr(output, attr)
                    break
            for attr in ["router_indices", "indices", "topk_indices", "selected_experts"]:
                if hasattr(output, attr) and torch.is_tensor(getattr(output, attr)):
                    indices = getattr(output, attr)
                    break

        if scores is None or indices is None:
            raise ValueError(
                f"Could not parse router output. type={type(output)}, "
                f"repr={repr(output)[:500]}"
            )

        return scores, indices

    def _normalize_router_output(self, router_scores, router_indices):
        scores = router_scores.detach()
        indices = router_indices.detach()

        if scores.dim() == 3:
            scores = scores.reshape(-1, scores.shape[-1])
        elif scores.dim() == 2:
            pass
        else:
            raise ValueError(f"Unexpected router_scores shape: {tuple(scores.shape)}")

        if indices.dim() == 3:
            indices = indices.reshape(-1, indices.shape[-1])
        elif indices.dim() == 2:
            pass
        else:
            raise ValueError(f"Unexpected router_indices shape: {tuple(indices.shape)}")

        return scores.float().cpu(), indices.long().cpu()

    def _make_hook(self, layer_idx):
        def hook(module, inputs, output):
            router_scores, router_indices = self._extract_router_tensors(output)
            scores, indices = self._normalize_router_output(router_scores, router_indices)

            num_tokens = scores.shape[0]
            self.token_counts[layer_idx] += num_tokens

            for t in range(num_tokens):
                selected = indices[t]
                selected_probs = scores[t, selected]

                for expert_id, prob in zip(selected.tolist(), selected_probs.tolist()):
                    self.selection_counts[layer_idx, expert_id] += 1
                    self.selection_prob_sums[layer_idx, expert_id] += float(prob)

        return hook

    def process_text(self, text):
        enc = self.tokenizer(
            text,
            return_tensors="pt",
            add_special_tokens=False,
        )
        enc = {k: v.to(self._input_device()) for k, v in enc.items()}
        self.text_lengths.append(int(enc["input_ids"].shape[1]))

        with torch.no_grad():
            _ = self.model(**enc, use_cache=False)

        self.num_texts += 1

    def build_result(self):
        total_input_tokens = int(sum(self.text_lengths))
        result = {
            "model_path": self.model_path,
            "texts_json_path": self.texts_json_path,
            "num_texts": int(self.num_texts),
            "num_layers": int(self.num_layers),
            "num_local_experts": int(self.num_experts),
            "total_input_tokens": total_input_tokens,
            "avg_input_tokens_per_text": (total_input_tokens / max(self.num_texts, 1)),
            "layer_stats": {},
        }

        for layer_idx in range(self.num_layers):
            counts = self.selection_counts[layer_idx].tolist()
            prob_sums = [float(x) for x in self.selection_prob_sums[layer_idx].tolist()]
            prob_means = [
                (prob_sums[i] / counts[i]) if counts[i] > 0 else 0.0
                for i in range(self.num_experts)
            ]

            ranking = sorted(
                range(self.num_experts),
                key=lambda e: (-counts[e], -prob_sums[e], e)
            )

            result[f"layer_{layer_idx}"] = ranking
            result["layer_stats"][f"layer_{layer_idx}"] = {
                "selection_counts": counts,
                "selection_prob_sums": prob_sums,
                "selection_prob_means": prob_means,
                "token_count": int(self.token_counts[layer_idx].item()),
                "selection_events": int(sum(counts)),
                "top_k_estimate": (
                    float(sum(counts)) / float(self.token_counts[layer_idx].item())
                    if int(self.token_counts[layer_idx].item()) > 0 else 0.0
                ),
            }

        return result


In [3]:
# Cell 3: recorder using MLP pre-hook, not router hook for MXFP4


class ExpertPatternRecorder:
    def __init__(self, model, tokenizer, model_path, texts_json_path):
        self.model = model
        self.tokenizer = tokenizer
        self.model_path = model_path
        self.texts_json_path = texts_json_path
        self.hooks = []

        self.layers = self.model.model.layers
        self.num_layers = len(self.layers)
        self.num_experts = int(self.model.config.num_local_experts)
        self.top_k = int(getattr(self.model.config, "num_experts_per_tok", 4))

        self.reset()

    def reset(self):
        self.selection_counts = torch.zeros(self.num_layers, self.num_experts, dtype=torch.long)
        self.selection_score_sums = torch.zeros(self.num_layers, self.num_experts, dtype=torch.float64)
        self.token_counts = torch.zeros(self.num_layers, dtype=torch.long)

        self.num_texts = 0
        self.text_lengths = []

    def _input_device(self):
        return next(self.model.parameters()).device

    def setup_hooks(self):
        self.remove_hooks()
        for layer_idx, layer in enumerate(self.layers):
            hook = layer.mlp.register_forward_pre_hook(self._make_mlp_prehook(layer_idx))
            self.hooks.append(hook)

    def remove_hooks(self):
        for h in self.hooks:
            h.remove()
        self.hooks = []

    def _make_mlp_prehook(self, layer_idx):
        def hook(module, inputs):
            hidden_states = inputs[0]
            router = module.router

            x = hidden_states.detach()
            if x.dim() == 3:
                x2d = x.reshape(-1, x.shape[-1])
            elif x.dim() == 2:
                x2d = x
            else:
                raise ValueError(f"Unexpected hidden_states shape: {tuple(x.shape)}")

            weight = router.weight.detach()
            bias = router.bias.detach() if router.bias is not None else None

            scores = x2d @ weight.transpose(0, 1)
            if bias is not None:
                scores = scores + bias

            topk_scores, topk_indices = torch.topk(scores, k=self.top_k, dim=-1)

            topk_indices_cpu = topk_indices.long().cpu()
            topk_scores_cpu = topk_scores.float().cpu()

            self.token_counts[layer_idx] += x2d.shape[0]

            for t in range(x2d.shape[0]):
                experts = topk_indices_cpu[t].tolist()
                vals = topk_scores_cpu[t].tolist()
                for e, s in zip(experts, vals):
                    self.selection_counts[layer_idx, e] += 1
                    self.selection_score_sums[layer_idx, e] += float(s)

        return hook

    def process_text(self, text):
        enc = self.tokenizer(
            text,
            return_tensors="pt",
            add_special_tokens=False,
        )
        enc = {k: v.to(self._input_device()) for k, v in enc.items()}
        self.text_lengths.append(int(enc["input_ids"].shape[1]))

        with torch.no_grad():
            _ = self.model(**enc, use_cache=False)

        self.num_texts += 1

    def build_result(self):
        total_input_tokens = int(sum(self.text_lengths))
        result = {
            "model_path": self.model_path,
            "texts_json_path": self.texts_json_path,
            "num_texts": int(self.num_texts),
            "num_layers": int(self.num_layers),
            "num_local_experts": int(self.num_experts),
            "num_experts_per_tok": int(self.top_k),
            "total_input_tokens": total_input_tokens,
            "avg_input_tokens_per_text": total_input_tokens / max(self.num_texts, 1),
            "layer_stats": {},
        }

        for layer_idx in range(self.num_layers):
            counts = self.selection_counts[layer_idx].tolist()
            score_sums = [float(x) for x in self.selection_score_sums[layer_idx].tolist()]
            score_means = [
                (score_sums[i] / counts[i]) if counts[i] > 0 else 0.0
                for i in range(self.num_experts)
            ]

            ranking = sorted(
                range(self.num_experts),
                key=lambda e: (-counts[e], -score_sums[e], e)
            )

            result[f"layer_{layer_idx}"] = ranking
            result["layer_stats"][f"layer_{layer_idx}"] = {
                "selection_counts": counts,
                "selection_score_sums": score_sums,
                "selection_score_means": score_means,
                "token_count": int(self.token_counts[layer_idx].item()),
                "selection_events": int(sum(counts)),
                "top_k_estimate": (
                    float(sum(counts)) / float(self.token_counts[layer_idx].item())
                    if int(self.token_counts[layer_idx].item()) > 0 else 0.0
                ),
            }

        return result




In [4]:

with open(TEXTS_JSON_PATH, "r") as f:
    texts = json.load(f)

assert isinstance(texts, list), "texts.json must be a JSON list"
assert len(texts) > 0, "texts.json is empty"

texts = [str(x) for x in texts]

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype="auto",
    device_map="auto",
    trust_remote_code=True,
)

recorder = ExpertPatternRecorder(
    model=model,
    tokenizer=tokenizer,
    model_path=MODEL_PATH,
    texts_json_path=TEXTS_JSON_PATH,
)

print("num_layers:", recorder.num_layers)
print("num_local_experts:", recorder.num_experts)
print("num_texts:", len(texts))


`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 15/15 [00:31<00:00,  2.12s/it]

num_layers: 36
num_local_experts: 128
num_texts: 6964


In [ ]:
# Cell 5: run collection

recorder.setup_hooks()

failed = []

try:
    for i, text in enumerate(tqdm(texts, desc="Collecting expert patterns")):
        try:
            recorder.process_text(text)
        except Exception as e:
            failed.append({"index": i, "error": str(e)})
finally:
    recorder.remove_hooks()

print("processed:", recorder.num_texts)
print("failed:", len(failed))
if failed:
    print(failed[:3])


In [ ]:
# Cell 6: save json

result = recorder.build_result()
result["failed"] = failed


saved: mappings/expert_mapping_120b_mxfp4.json
layer_0 top 10: [49, 113, 60, 17, 15, 77, 45, 123, 99, 100]
layer_1 top 10: [117, 12, 99, 11, 109, 102, 60, 107, 10, 63]


In [ ]:
# Cell 6.5: print selection counts in descending order per layer

def print_expert_counts(result, top_n=20, layers=None):
    """
    top_n: how many experts to show per layer
    layers: list of layer indices to print, e.g. [0, 1, 10]. None = all layers
    """
    num_layers = result["num_layers"]
    layer_indices = layers if layers is not None else range(num_layers)

    for layer_idx in layer_indices:
        stats = result["layer_stats"][f"layer_{layer_idx}"]
        counts = stats["selection_counts"]
        total_events = stats["selection_events"]
        token_count = stats["token_count"]

        # pair (expert_idx, count) and sort descending
        ranked = sorted(enumerate(counts), key=lambda x: -x[1])

        print(f"\n── Layer {layer_idx} ──  tokens={token_count}  total_selections={total_events}")
        print(f"  {'Rank':<6} {'Expert':>8} {'Count':>10} {'% of selections':>18} {'Cumulative %':>14}")
        print(f"  {'-'*60}")

        cumulative = 0
        for rank, (expert_idx, count) in enumerate(ranked[:top_n]):
            pct = 100.0 * count / total_events if total_events > 0 else 0.0
            cumulative += pct
            print(f"  {rank:<6} {expert_idx:>8} {count:>10} {pct:>17.2f}% {cumulative:>13.2f}%")

        # also show the bottom
        zero_count = sum(1 for c in counts if c == 0)
        never_used = [e for e, c in enumerate(counts) if c == 0]
        print(f"  ... ({len(counts) - top_n} more experts not shown)")
        print(f"  Experts never activated: {zero_count} {never_used[:10]}{'...' if len(never_used) > 10 else ''}")


# print only a few layers to get a feel, e.g. first, middle, last
sample_layers = [0, 1, result["num_layers"] // 2, result["num_layers"] - 1]
print_expert_counts(result, top_n=20, layers=sample_layers)

# or print ALL layers
# print_expert_counts(result, top_n=20, layers=None)

In [ ]:

output_path = Path(OUTPUT_JSON_PATH)
output_path.parent.mkdir(parents=True, exist_ok=True)

print("saved:", str(output_path))
print("layer_0 top 10:", result["layer_0"][:10])
print("layer_1 top 10:", result["layer_1"][:10])


In [8]:

with open(output_path, "w") as f:
    json.dump(result, f, indent=2)



In [9]:

# Cell 7: quick smoke check of saved file

with open(OUTPUT_JSON_PATH, "r") as f:
    saved = json.load(f)

print(saved["model_path"])
print(saved["num_texts"], saved["num_layers"], saved["num_local_experts"])
print(saved["layer_0"][:10])
print(saved["layer_stats"]["layer_0"]["top_k_estimate"])


/data/aneesh/pruning/oss_120b_mxfp4
28 36 128
[49, 113, 60, 17, 15, 77, 45, 123, 99, 100]
4.0
